# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 1, 1.3333
2. 2
3. 3
4. 1005
5. [9, 10]

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [2]:
# 1
import pandas as pd
def clean(path):
  df = pd.read_csv(path)
  df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
  print("После приведения amount:", len(df))

  city_map = {
    "almaty": "Алматы",
    "astana": "Астана",
    "shymkent": "Шымкент",
}
  df["city"] = df["city"].str.strip().str.lower().replace(city_map).str.title()
  print("После нормализации city:", len(df))

  df = df.drop_duplicates()
  print("После удаления дублей:", len(df))

  df["order_date"] = pd.to_datetime(df["order_date"])
  print("После приведения дат:", len(df))

  df = df[df["customer_id"] != "TEST"]
  print("После удаления TEST:", len(df))

  # оператор самопроверки: он проверяет, истинно ли условие после него,
  # и если условие ложно (False), немедленно останавливает программу и выбрасывает ошибку AssertionError
  assert df["amount"].isna().sum() == 0
  assert df["city"].isna().sum() == 0
  assert df["order_id"].nunique() == len(df)

  return df

df = clean("orders.csv")

После приведения amount: 5463
После нормализации city: 5463
После удаления дублей: 5283
После приведения дат: 5283
После удаления TEST: 5280


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [3]:
# 2
paid = df[df["status"] == "оплачен"]
total_sep = paid[paid["order_date"].dt.month == 9]["amount"].sum()
total_oct = paid[paid["order_date"].dt.month == 10]["amount"].sum()
print(f"Выручка в сентябре: {total_sep}")
print(f"Выручка в октябре: {total_oct}")
print(f"Изменение выручки в процентах: {((total_oct - total_sep) / total_sep * 100).round(3)}%")
print()
for city in (df['city'].unique()):
  amount_sep = paid[(paid['city'] == city) & (paid["order_date"].dt.month == 9)]["amount"].sum()
  amount_oct = paid[(paid['city'] == city) & (paid["order_date"].dt.month == 10)]["amount"].sum()
  diff = (amount_oct - amount_sep) / amount_sep * 100
  print(f"Изменение выручки в процентах {city:<9}: {diff:>7.3f}%")

# здесь мы брали только оплаченные заказы после очистки;
# если в таблице урока 4 учитывались все заказы и не применялась часть шагов очистки
# падает сильнее всех город Шымкент

Выручка в сентябре: 35243410.0
Выручка в октябре: 34419520.0
Изменение выручки в процентах: -2.338%

Изменение выручки в процентах Алматы   :   0.979%
Изменение выручки в процентах Астана   :  -2.033%
Изменение выручки в процентах Караганда:  -7.588%
Изменение выручки в процентах Шымкент  : -11.289%
Изменение выручки в процентах Атырау   :   4.964%
Изменение выручки в процентах Актобе   :  -1.418%


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

| Шаг очистки | Убрано строк | Осталось строк | Изменение выручки (сент. → окт.) |
| :--- | :---: | :---: | :---: |
| Исходный файл `orders.csv` | — | 5463 | −22,7% |
| 1. Приведение `amount` к числам | 0 | 5463 | −22,7% |
| 2. Нормализация `city` через словарь | 0 | 5463 | −22,7% |
| 3. Удаление дублей (`drop_duplicates()`) | 180 | 5283 | −19,6% |
| 4. Приведение `order_date` к дате | 0 | 5283 | −19,6% |
| 5. Удаление тестовых записей (`TEST`) | 3 | 5280 | −2,3% |

**Для менеджера:** сильнее всего цифру исказило удаление тестовых заказов (`TEST`) — две сентябрьские тестовые строки по `9 999 990` сильно завышали базу сентября и создавали видимость падения на `−22,7%` вместо реальных `−2,3%`.

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [4]:
# 4
print(f"Количество пропусков в delivery_days: {df["delivery_days"].isna().sum()}")
for city in (df['city'].unique()):
  for status in (df['status'].unique()):
    print(f"Количество пропусков в delivery_days для города {city} и статуса {status}: {df[(df['city'] == city) & (df['status'] == status)]['delivery_days'].isna().sum()}")

# Больше всего в городе Алматы и статус оплачен

Количество пропусков в delivery_days: 158
Количество пропусков в delivery_days для города Алматы и статуса оплачен: 61
Количество пропусков в delivery_days для города Алматы и статуса отменён: 2
Количество пропусков в delivery_days для города Астана и статуса оплачен: 35
Количество пропусков в delivery_days для города Астана и статуса отменён: 5
Количество пропусков в delivery_days для города Караганда и статуса оплачен: 23
Количество пропусков в delivery_days для города Караганда и статуса отменён: 1
Количество пропусков в delivery_days для города Шымкент и статуса оплачен: 17
Количество пропусков в delivery_days для города Шымкент и статуса отменён: 5
Количество пропусков в delivery_days для города Атырау и статуса оплачен: 4
Количество пропусков в delivery_days для города Атырау и статуса отменён: 1
Количество пропусков в delivery_days для города Актобе и статуса оплачен: 4
Количество пропусков в delivery_days для города Актобе и статуса отменён: 0


In [5]:
sh = df[df["city"] == "Шымкент"]
share_dropna = (sh["delivery_days"].dropna() == 2).mean() * 100
median_val = df["delivery_days"].median()
share_median = (sh["delivery_days"].fillna(median_val) == 2).mean() * 100
share_zero = (sh["delivery_days"].fillna(0) == 2).mean() * 100
print(f"Доля доставки за 2 дня в Шымкенте (dropna): {share_dropna:.2f}%")
print(f"Доля доставки за 2 дня в Шымкенте (fillna(медиана)): {share_median:.2f}%")
print(f"Доля доставки за 2 дня в Шымкенте (fillna(0)): {share_zero:.2f}%")

# Честный вариант Он считает долю доставок за 2 дня только по тем заказам,
# где факт и срок доставки действительно зафиксированы, и не придумывает несуществующие данные.

Доля доставки за 2 дня в Шымкенте (dropna): 14.85%
Доля доставки за 2 дня в Шымкенте (fillna(медиана)): 17.27%
Доля доставки за 2 дня в Шымкенте (fillna(0)): 14.43%


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [9]:
# 5
ds = pd.read_csv("orders.csv")
ds["city"] = ds["city"].str.lower().str.strip().replace({
    "almaty": "Алматы",
    "astana": "Астана",
    "shymkent": "Шымкент",
}).str.title()
ds = ds.drop_duplicates()
print(ds["city"].nunique(), len(ds))

6 5283


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [15]:
# 6
day_orders = df["order_date"].dt.day_name().value_counts()
print(day_orders)
print(f"День с наименьшим количеством заказов: {day_orders.idxmin()} ({day_orders.min()} заказов)")
print(f"День с наибольшим количеством заказов: {day_orders.idxmax()} ({day_orders.max()} заказов)")
difference = (day_orders.max() - day_orders.min()) / day_orders.max() * 100
print(f"Разница в процентах: {difference:.2f}%")

# Понедельник базовый, спокойный график. В эти дни можно ставить меньше смен
# Пиковые дни (вторник–пятница, 781–786 заказов, и суббота - 773 заказа): нагрузка стабильно выше примерно на 13% по сравнению с понедельником.
# На эти дни (особенно на четверг, вторник и пятницу)
# стоит выводить максимум курьеров и делать усиление в часы пик, чтобы не росло время доставки и не копился заказы.

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
День с наименьшим количеством заказов: Monday (684 заказов)
День с наибольшим количеством заказов: Thursday (786 заказов)
Разница в процентах: 12.98%


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8